# Проверка снимков SCD2

Ноутбук предназначен для проверки результатов DAG после трёх scheduled-запусков. Он не запускает расчётные задачи и не меняет данные в S3.

In [1]:
from jobs.common import create_spark_session, dataset_name, s3_uri
from pyspark.sql import functions as F

spark = create_spark_session("hw23-notebook-scd2-check")
dataset = dataset_name()
dates = ["2026-01-23", "2026-01-24", "2026-01-25"]


26/11/20 20:49:20 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


### Загрузка снимков

In [2]:
snapshots = {
    date: spark.read.parquet(
        s3_uri(f"results/{dataset}/user_scd2/snapshot_date={date}")
    )
    for date in dates
}

df_scd2 = snapshots[dates[-1]]
df_scd2.printSchema()
df_scd2.orderBy("user_id", "valid_from").show(30, truncate=False)


26/11/20 20:49:21 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties


root
 |-- user_id: long (nullable = true)
 |-- user_fio: string (nullable = true)
 |-- user_email: string (nullable = true)
 |-- user_login: string (nullable = true)
 |-- valid_from: timestamp (nullable = true)
 |-- valid_to: timestamp (nullable = true)
 |-- is_current: boolean (nullable = true)
 |-- snapshot_date: date (nullable = true)
 |-- _dag_id: string (nullable = true)
 |-- _run_id: string (nullable = true)
 |-- _task_id: string (nullable = true)
 |-- _data_interval_start: timestamp (nullable = true)
 |-- _data_interval_end: timestamp (nullable = true)
 |-- _loaded_at: timestamp (nullable = true)

+-------+----------------+------------------------------------+-----------+-------------------+-------------------+----------+-------------+------------------------+------------------------------------+---------------+--------------------+-------------------+--------------------------+
|user_id|user_fio        |user_email                          |user_login |valid_from         |valid_

### Сводка по датам

In [3]:
for date, df in snapshots.items():
    print(f"\nДата снимка: {date}")
    print("Строк:", df.count())
    print("Пользователей:", df.select("user_id").distinct().count())
    df.groupBy("is_current").count().orderBy("is_current").show()



Дата снимка: 2026-01-23
Строк: 961
Пользователей: 100
+----------+-----+
|is_current|count|
+----------+-----+
|     false|  861|
|      true|  100|
+----------+-----+


Дата снимка: 2026-01-24
Строк: 995
Пользователей: 100
+----------+-----+
|is_current|count|
+----------+-----+
|     false|  895|
|      true|  100|
+----------+-----+


Дата снимка: 2026-01-25
Строк: 1036
Пользователей: 100
+----------+-----+
|is_current|count|
+----------+-----+
|     false|  936|
|      true|  100|
+----------+-----+



### Проверка ключей и текущих версий

In [4]:
for date, df in snapshots.items():
    duplicate_keys = (
        df.groupBy("user_id", "valid_from").count()
        .filter(F.col("count") > 1).count()
    )
    all_users = df.select("user_id").distinct()
    current_counts = (
        df.filter(F.col("is_current"))
        .groupBy("user_id").count()
        .withColumnRenamed("count", "current_count")
    )
    current_per_user = (
        all_users.join(current_counts, "user_id", "left")
        .filter(F.coalesce(F.col("current_count"), F.lit(0)) != 1)
        .count()
    )
    invalid_current = df.filter(
        (F.col("is_current") & F.col("valid_to").isNotNull()) |
        ((~F.col("is_current")) & F.col("valid_to").isNull())
    ).count()
    print(date, {
        "duplicate_keys": duplicate_keys,
        "users_without_exactly_one_current_version": current_per_user,
        "invalid_current_flags": invalid_current,
    })
    assert duplicate_keys == 0
    assert current_per_user == 0
    assert invalid_current == 0


2026-01-23 {'duplicate_keys': 0, 'users_without_exactly_one_current_version': 0, 'invalid_current_flags': 0}
2026-01-24 {'duplicate_keys': 0, 'users_without_exactly_one_current_version': 0, 'invalid_current_flags': 0}
2026-01-25 {'duplicate_keys': 0, 'users_without_exactly_one_current_version': 0, 'invalid_current_flags': 0}


### Проверка непересечения полуоткрытых интервалов

In [5]:
from pyspark.sql.window import Window

for date, df in snapshots.items():
    w = Window.partitionBy("user_id").orderBy("valid_from")
    checked = df.withColumn("_next_valid_from", F.lead("valid_from").over(w))
    overlaps = checked.filter(
        F.col("valid_to").isNull() & F.col("_next_valid_from").isNotNull() |
        (F.col("valid_to") > F.col("_next_valid_from"))
    ).count()
    gaps_or_mismatches = checked.filter(
        F.col("_next_valid_from").isNotNull() &
        (F.col("valid_to") != F.col("_next_valid_from"))
    ).count()
    print(date, "пересечения:", overlaps, "несмежные границы:", gaps_or_mismatches)
    assert overlaps == 0
    assert gaps_or_mismatches == 0


2026-01-23 пересечения: 0 несмежные границы: 0
2026-01-24 пересечения: 0 несмежные границы: 0
2026-01-25 пересечения: 0 несмежные границы: 0


### Проверка повторов без изменения атрибутов

In [6]:
# Вспомогательная выборка: соседние версии не должны иметь одинаковые атрибуты.
for date, df in snapshots.items():
    w = Window.partitionBy("user_id").orderBy("valid_from")
    checked = (
        df.withColumn("_prev_fio", F.lag("user_fio").over(w))
          .withColumn("_prev_email", F.lag("user_email").over(w))
          .withColumn("_prev_login", F.lag("user_login").over(w))
    )
    redundant = checked.filter(
        (F.col("user_fio") == F.col("_prev_fio")) &
        (F.col("user_email") == F.col("_prev_email")) &
        (F.col("user_login") == F.col("_prev_login"))
    ).count()
    print(date, "избыточных соседних версий:", redundant)
    assert redundant == 0


2026-01-23 избыточных соседних версий: 0
2026-01-24 избыточных соседних версий: 0
2026-01-25 избыточных соседних версий: 0
